In [2]:
import sys,os
sys.path.append(r'X:/Strategies/LeadLagXGB/')
sys.path.append(r'Z:/EnergyTrading/Python/')
sys.path.append(r'Z:/EnergyTrading/Python/Strategies/LeadLagXGB/')
from support_functions import calculate_MACD, calculate_lead_lag_triggers, calculate_regression_model_price, calc_vol_intensity_index

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.DB_reader import Database
from datetime import date, timedelta

# from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI

# from Strategies.LeadLagXGB.backtest_class import BacktestLL
# from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass
tol=(1e-1)/2

In [4]:
def get_trades_for_contract(contract, start_date, end_date):
    params_dict={}
    params_dict['tenor_list'] = ['dec'] if contract=='euadec1' else [contract[-2]]
    params_dict['tn1_list'] = [int(contract[-1])]
    params_dict['mkt_list'] = ['eua'] * len(params_dict['tenor_list']) if contract=='euadec1' else [contract[0:-2]] * len(params_dict['tenor_list'])
    params_dict['tn2_list'] = []
    params_dict['prod'] = 'base'
    params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
    params_dict['start_date'] = start_date
    params_dict['end_date'] = end_date
    params_dict['ns'] = 2

    # Fetch trades and best orders for the curve
    assembler = TPDataAssembly(source='trayport', user='matej')
    # assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
    trades_dict = assembler.get_data(params_dict, target_data='trades')
    #assembler.set_data_source('database')
    #ba_dict = assembler.get_data(params_dict, target_data='best_orders')


    trades = pd.DataFrame()
    products = []
    for key in trades_dict.keys():
        trade_aux = trades_dict[key].copy()
        trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
        if trades.empty:
            trades = trade_aux.copy()
        else:
            trades = pd.concat([trades, trade_aux])
        products.append(key)
    trades.sort_index(inplace=True)




    data_raw = trades
    print(data_raw.columns)
    data_raw['tradeid_'+contract]=data_raw['tradeid_'+contract].apply(lambda x: str(x)[:-7] if str(x)[-7:]==' Public' else str(x))
    df_lead = data_raw[data_raw['broker_id_'+contract]==1441][['tradeid_'+contract,'price_'+contract, 'volume_'+contract]].copy()
    
    df_lead['contract']=contract

    # data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
    #                    'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()

    df_lead.columns = [a.split('_')[0] for a in df_lead.columns]
    print(df_lead.columns)
    df_lead.columns = ['tradeid', 'trd_price', 'volume', 'contract']
    
    return df_lead

# BASIC TARGETS - N-trades ahead price moved by X%

In [5]:
## Selecting all trades since 2024

# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='2025-01-01' and datetime<='2025-05-29' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710', '10001075', '10100480', '10012528', '10002806')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 2072109 rows from source database.


In [6]:
contracts=['dem1', 'dem2', 'deq1', 'dey1']#, 'frm1', 'frq1', 'fry1', 'ttfm1'
contracts

['dem1', 'dem2', 'deq1', 'dey1']

In [7]:
start_date=(date.today()+timedelta(-146)).isoformat()
end_date=(date.today()+timedelta(-1)).isoformat()

print(start_date)
print(end_date)

2025-01-07
2025-06-01


In [8]:
df_all=pd.concat([get_trades_for_contract(contract, start_date, end_date) for contract in contracts])

https://referencedata.trayport.com/instruments
Duration: 1.182s
https://analytics.trayport.com/api/trades?from=2025-01-07T07%3A00%3A00Z&until=2025-01-29T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=254&ContractType=SinglePeriod
Duration: 1.434s
https://referencedata.trayport.com/instruments
Duration: 0.565s
https://analytics.trayport.com/api/trades?from=2025-01-30T07%3A00%3A00Z&until=2025-02-26T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=255&ContractType=SinglePeriod
Duration: 1.912s
https://referencedata.trayport.com/instruments
Duration: 0.510s
https://analytics.trayport.com/api/trades?from=2025-02-27T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=256&ContractType=SinglePeriod
Duration: 1.293s
https://referencedata.trayport.com/instruments
Duration: 0.700s
https://analytics.trayport.com/api/trades?from=2025-03-28T07%3A00%3A00Z&until=2025-04-28T18%3A00%3A00Z&instrumentId=1064171

In [9]:
df_all['contract'].value_counts()

contract
dem1    304119
dem2    104656
deq1     90131
dey1     53366
Name: count, dtype: int64

In [10]:
df_all=df_all.reset_index()

In [11]:
df_all['datetime'] = pd.to_datetime(df_all['datetime'], errors='coerce')
df_all['timestamp'] = df_all['datetime']

In [12]:
df_all=df_all[df_all['datetime'].apply(lambda x: x.hour>=8 and  x.hour<18)]

In [13]:
df_all.head()

,datetime,tradeid,trd_price,volume,contract,timestamp
0,2025-01-07 08:00:22.059673786,Eurex T7/DEBM022025-20250107/2/1,106.99,1,dem1,2025-01-07 08:00:22.059673786
1,2025-01-07 08:00:48.891798019,Eurex T7/DEBM022025-20250107/3/1,106.50,1,dem1,2025-01-07 08:00:48.891798019
2,2025-01-07 08:01:25.213961363,Eurex T7/DEBM022025-20250107/10/1,106.50,1,dem1,2025-01-07 08:01:25.213961363
3,2025-01-07 08:01:25.214008093,Eurex T7/DEBM022025-20250107/11/3,106.50,3,dem1,2025-01-07 08:01:25.214008093
4,2025-01-07 08:01:25.266085385,Eurex T7/DEBM022025-20250107/12/1,106.50,1,dem1,2025-01-07 08:01:25.266085385


In [14]:
N = 10  # number of trades to look ahead
X = 0.02  # 1% price move

def calc_targets(group, N, X, contract):
    """
    Calculate target_up and target_down for each trade in the group.
    target_up is True if any of the next N trades have a price >= current price * (1 + X).
    target_down is True if any of the next N trades have a price <= current price * (1 - X).
    """
    prices = group['trd_price'].values
    target_up = []
    target_down = []
    for i, price in enumerate(prices):
        if i + N < len(prices):
            future_prices = prices[i+1:i+N+1]
            target_up.append(int(np.any(future_prices >= price * (1 + X))))
            target_down.append(int(np.any(future_prices <= price * (1 - X))))
        else:
            target_up.append(np.nan)
            target_down.append(np.nan)
    group[f'target_long_{contract}_{N}_trds_{X*100}_perc'] = target_up
    group[f'target_short_{contract}_{N}_trds_{X*100}_perc'] = target_down
    return group

Ns = [10, 20]
Xs = [0.01, 0.03, 0.05]

for n in Ns:
    for x in Xs:
        df_all = df_all.groupby('contract', group_keys=False).apply(
            lambda df: calc_targets(df, n, x, df['contract'].iloc[0])
        )


C:\Users\andrej\AppData\Local\Temp\ipykernel_51900\2299761210.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all = df_all.groupby('contract', group_keys=False).apply(
C:\Users\andrej\AppData\Local\Temp\ipykernel_51900\2299761210.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all = df_all.groupby('contract', group_keys=False).apply(
C:\Users\andrej\AppData\Local\Temp\ipykernel_51900\2299761

In [15]:
df_all.head(10)

,datetime,tradeid,trd_price,volume,contract,timestamp,target_long_dem1_10_trds_1.0_perc,target_short_dem1_10_trds_1.0_perc,target_long_dem2_10_trds_1.0_perc,target_short_dem2_10_trds_1.0_perc,...,target_long_dey1_20_trds_3.0_perc,target_short_dey1_20_trds_3.0_perc,target_long_dem1_20_trds_5.0_perc,target_short_dem1_20_trds_5.0_perc,target_long_dem2_20_trds_5.0_perc,target_short_dem2_20_trds_5.0_perc,target_long_deq1_20_trds_5.0_perc,target_short_deq1_20_trds_5.0_perc,target_long_dey1_20_trds_5.0_perc,target_short_dey1_20_trds_5.0_perc
0,2025-01-07 08:00:22.059673786,Eurex T7/DEBM022025-20250107/2/1,106.99,1,dem1,2025-01-07 08:00:22.059673786,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-07 08:00:48.891798019,Eurex T7/DEBM022025-20250107/3/1,106.50,1,dem1,2025-01-07 08:00:48.891798019,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-07 08:01:25.213961363,Eurex T7/DEBM022025-20250107/10/1,106.50,1,dem1,2025-01-07 08:01:25.213961363,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-07 08:01:25.214008093,Eurex T7/DEBM022025-20250107/11/3,106.50,3,dem1,2025-01-07 08:01:25.214008093,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01-07 08:01:25.266085385,Eurex T7/DEBM022025-20250107/12/1,106.50,1,dem1,2025-01-07 08:01:25.266085385,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2025-01-07 08:01:25.266109704,Eurex T7/DEBM022025-20250107/13/1,106.50,1,dem1,2025-01-07 08:01:25.266109704,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
6,2025-01-07 08:01:25.266122340,Eurex T7/DEBM022025-20250107/14/2,106.50,2,dem1,2025-01-07 08:01:25.266122340,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
7,2025-01-07 08:01:38.864053965,Eurex T7/DEBM022025-20250107/15/1,106.26,1,dem1,2025-01-07 08:01:38.864053965,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
8,2025-01-07 08:01:38.976566315,Eurex T7/DEBM022025-20250107/16/1,106.26,1,dem1,2025-01-07 08:01:38.976566315,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
9,2025-01-07 08:01:39.187114239,Eurex T7/DEBM022025-20250107/17/3,106.25,3,dem1,2025-01-07 08:01:39.187114239,0.0,0.0,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# conn = Database('timescaledb')
# conn._connect()

# # Find all target columns (those starting with 'target_long_' or 'target_short_')
# target_cols = [col for col in df_all.columns if col.startswith('target_long_') or col.startswith('target_short_')]

# for col in target_cols:
#     contract = col.split('_')[2]  # Extract contract from column name
#     sql = f"""
# INSERT INTO public.targets (target_name, main_contract, description, location, author, additional)
# VALUES
#     ('{col}', '{contract}', 'Auto-generated target column looking N EEX trades ahead and checking if price moved by X percent, in the sense any of those N trades reached this level', 'Projects/EnergyTrading/Python/Strategies/Targets/Targets_N_trds_ahead_X_perc.ipynb', 'AndrejZubal', 'Type: N_trds_ahead_X_percent')
# ON CONFLICT (target_id) DO NOTHING;
# """
#     print(sql)
#     conn.execute_general_query(sql)
    

In [17]:
target_id_dict = {
    "target_long_dem1_10_trds_1.0_perc": 121,
    "target_short_dem1_10_trds_1.0_perc": 122,
    "target_long_dem2_10_trds_1.0_perc": 123,
    "target_short_dem2_10_trds_1.0_perc": 124,
    "target_long_deq1_10_trds_1.0_perc": 125,
    "target_short_deq1_10_trds_1.0_perc": 126,
    "target_long_dey1_10_trds_1.0_perc": 127,
    "target_short_dey1_10_trds_1.0_perc": 128,

    "target_long_dem1_10_trds_3.0_perc": 129,
    "target_short_dem1_10_trds_3.0_perc": 130,
    "target_long_dem2_10_trds_3.0_perc": 131,
    "target_short_dem2_10_trds_3.0_perc": 132,
    "target_long_deq1_10_trds_3.0_perc": 133,
    "target_short_deq1_10_trds_3.0_perc": 134,
    "target_long_dey1_10_trds_3.0_perc": 135,
    "target_short_dey1_10_trds_3.0_perc": 136,

    "target_long_dem1_10_trds_5.0_perc": 137,
    "target_short_dem1_10_trds_5.0_perc": 138,
    "target_long_dem2_10_trds_5.0_perc": 139,
    "target_short_dem2_10_trds_5.0_perc": 140,
    "target_long_deq1_10_trds_5.0_perc": 141,
    "target_short_deq1_10_trds_5.0_perc": 142,
    "target_long_dey1_10_trds_5.0_perc": 143,
    "target_short_dey1_10_trds_5.0_perc": 144,

    "target_long_dem1_20_trds_1.0_perc": 145,
    "target_short_dem1_20_trds_1.0_perc": 146,
    "target_long_dem2_20_trds_1.0_perc": 147,
    "target_short_dem2_20_trds_1.0_perc": 148,
    "target_long_deq1_20_trds_1.0_perc": 149,
    "target_short_deq1_20_trds_1.0_perc": 150,
    "target_long_dey1_20_trds_1.0_perc": 151,
    "target_short_dey1_20_trds_1.0_perc": 152,

    "target_long_dem1_20_trds_3.0_perc": 153,
    "target_short_dem1_20_trds_3.0_perc": 154,
    "target_long_dem2_20_trds_3.0_perc": 155,
    "target_short_dem2_20_trds_3.0_perc": 156,
    "target_long_deq1_20_trds_3.0_perc": 157,
    "target_short_deq1_20_trds_3.0_perc": 158,
    "target_long_dey1_20_trds_3.0_perc": 159,
    "target_short_dey1_20_trds_3.0_perc": 160,

    "target_long_dem1_20_trds_5.0_perc": 161,
    "target_short_dem1_20_trds_5.0_perc": 162,
    "target_long_dem2_20_trds_5.0_perc": 163,
    "target_short_dem2_20_trds_5.0_perc": 164,
    "target_long_deq1_20_trds_5.0_perc": 165,
    "target_short_deq1_20_trds_5.0_perc": 166,
    "target_long_dey1_20_trds_5.0_perc": 167,
    "target_short_dey1_20_trds_5.0_perc": 168
}


In [ ]:
target_cols = [col for col in df_all.columns if col.startswith('target_long_') or col.startswith('target_short_')]

# Prepare targets DataFrame
targets = df_all[['datetime', 'tradeid', 'contract'] + target_cols].copy()

df_merged = df.merge(
targets[['tradeid']+target_cols],
on='tradeid', how='inner'
)

# Melt to long format for all targets
targets_long = df_merged.melt(
    id_vars=['datetime', 'nanotime', 'tradeid'],
    value_vars=target_cols,
    var_name='target_name',
    value_name='target_value'
)
targets_long['target_id'] = targets_long['target_name'].map(target_id_dict)  # or use a mapping dict if needed
#targets_long['additional'] = None  # set accordingly if you have additional data

# Step 3: Sort by datetime for proper forward filling
targets_long = targets_long.sort_values(by=['datetime', 'nanotime','target_id'], ascending=[True, True, True])

targets_long.dropna(subset=['target_value'])

targets_long=targets_long[['datetime', 'nanotime', 'tradeid', 'target_id','target_value']]
# Step 4: Forward fill within each day separately
#targets_long['date_only'] = targets_long['datetime'].dt.date
#targets_long['target_value'] = targets_long.groupby(['target_id', 'date_only'])['target_value'].ffill()    

# Locate rows where pred_value is missing and set additional message
#mask_missing = targets_long['target_value'].isna()
#targets_long.loc[mask_missing, 'additional'] = 'Not enough observations to calculate the target_value!'

#del targets_long['date_only']  # Clean up temporary column

# Connect to TimescaleDB
batch_size = 1_000_000
conn = Database('timescaledb')
conn._connect()

# Insert in batches
total_rows = len(targets_long)
print(total_rows)
for start in range(0, total_rows, batch_size):
    end_idx = min(start + batch_size, total_rows)
    batch = targets_long.iloc[start:end_idx]

    # Insert into targets_dataset_entries
    batch.to_sql('targets_dataset_entries', conn.engine, schema='public', index=False, if_exists='append', method='multi')
    print(f"✅ Inserted rows {start} to {end_idx} into targets_dataset_entries.")

print("🎉 All batches inserted successfully.")

#     send_plain_email(
#         RECIPIENT,
#         "SUCCESS: FairPriceCalculatorForDatamart_daily_update job",
#         f'{end_date} number of records: {total_rows}',
#         email_password=EMAIL_PASSWORD
#     )
#     print('\n')

# except Exception as e:
#     send_plain_email(
#         RECIPIENT,
#         "FAIL: FairPriceCalculatorForDatamart_daily_update",
#         f'The upload of data failed for this run, please check what is the issue.\nError: {e}',
#         email_password=EMAIL_PASSWORD
#     )

Connected to the database timescaledb
25990896
